# Dataset Generator from PDF Manuals

This notebook extracts text from a PDF manual, splits it into semantic chunks, and generates question-answer (QA) pairs using a local LLM (via Ollama). The resulting dataset can be used for fine-tuning or evaluation.

## Overview
1. **Extract text** from PDF pages.
2. **Chunk text** into semantic sections (respecting natural boundaries).
3. **Filter valid chunks** (minimum length, no noise).
4. **Generate QA pairs** using a language model with diverse question types.
5. **Verify consistency** of each QA pair.
6. **Remove near-duplicate questions**.
7. **Save results** as JSONL.

Let's start by importing the required libraries.

In [ ]:
import pdfplumber
import ollama
import json
from tqdm import tqdm

## Main Code: Configuration, Extraction, Chunking, QA Generation

Below is the complete pipeline. All code comments and print messages have been translated to English for clarity.

In [ ]:
import json
import re
import random
import pdfplumber
import ollama
from tqdm import tqdm
from dataclasses import dataclass, asdict
from typing import Optional, List, Dict

# --- Centralized configuration ---
MIN_CHUNK_CHARS = 100
MAX_CHUNK_CHARS = 400
MIN_CHUNK_WORDS = 20          # Reduced to keep more chunks
MAX_WORDS_QA_TARGET = 12      # Ideal target
MAX_WORDS_QA_ACCEPT = 15      # Acceptable to avoid excessive discarding

@dataclass
class QAPair:
    question: str
    answer: str
    source_chunk: str
    page_hint: str
    confidence: str  # "high" | "low" | "rejected"

# 1. Extraction with page metadata
def extract_text_from_pdf(pdf_path: str) -> List[dict]:
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            text = page.extract_text()
            if text and text.strip():
                pages.append({"text": text.strip(), "page": i + 1})
    return pages

# 2. Semantic chunking by sections
def split_into_chunks(pages: List[dict]) -> List[dict]:
    chunks = []
    buffer_text = ""
    buffer_page = 1

    section_pattern = re.compile(
        r"^\s*(\d+[\.\d]*\s+[A-Z]|CHAPTER\s+\d+|Seção\s+\d+|Section\s+\d+)",
        re.MULTILINE | re.IGNORECASE
    )

    for page_data in pages:
        text = page_data["text"]
        page = page_data["page"]

        parts = section_pattern.split(text)
        for part in parts:
            part = part.strip()
            if not part:
                continue

            if len(buffer_text) + len(part) > MAX_CHUNK_CHARS and len(buffer_text) >= MIN_CHUNK_CHARS:
                chunks.append({"text": buffer_text.strip(), "page": buffer_page})
                buffer_text = part
                buffer_page = page
            else:
                buffer_text += "\n" + part
                if not buffer_text.strip():
                    buffer_page = page

    if buffer_text.strip() and len(buffer_text) >= MIN_CHUNK_CHARS:
        chunks.append({"text": buffer_text.strip(), "page": buffer_page})

    return chunks

# 3. Chunk quality filter (less restrictive)
def is_valid_chunk(chunk_text: str) -> bool:
    word_count = len(chunk_text.split())
    if word_count < MIN_CHUNK_WORDS:
        return False
    noise_patterns = [
        r"^\s*\d+\s*$",
        r"^(fig|figure|table|figura)\s*\d",
        r"^©",
        r"^(contents|índice|index)\s*$",
    ]
    for pattern in noise_patterns:
        if re.match(pattern, chunk_text.strip(), re.IGNORECASE):
            return False
    return True

# 4. QA generation with diversity and word tolerance
def generate_qa(chunk: dict, model: str = "qwen3-coder", temperature: float = 0.9) -> tuple[Optional[str], Optional[str]]:
    question_types = [
        "quantity (How many...?)",
        "location (Where is...?)",
        "procedure step (What is next after...?)",
        "condition (When should...?)",
        "safety (Why must...?)",
        "specification (Max...?)",
        "function (What does X do?)",
        "cause (What happens if...?)",
        "identification (Which part...?)",
        "time (How long...?)",
        "value (Torque for...?)",
        "comparison (Difference between X and Y?)",
        "troubleshooting (Cause of error X?)",
    ]
    random.shuffle(question_types)
    types_str = ", ".join(question_types[:5])

    prompt = f"""You are building a technical QA dataset from a product manual.
Create ONE question and its answer FROM THE CONTENT BELOW.

⚠️ STRICT REQUIREMENTS:
- Question: aim for ≤12 words, MUST be ≤15 words.
- Answer: aim for ≤12 words, MUST be ≤15 words.
- Question must be specific, technical, and answerable only from the content.
- Avoid vague questions like "What is described?".
- Use varied structures: "How many...", "Where is...", "When should...", etc.

🎲 Inspiration types: {types_str}

Examples of GOOD questions (≤12 words):
- "How many screws secure the front panel?"
- "Where is the emergency stop button located?"
- "When should the filter be replaced?"
- "What torque is required for M8 bolts?"

FORMAT EXACTLY AS:
INSTRUCTION: <question (max 15 words)>
RESPONSE: <answer (max 15 words)>

Technical content (page {chunk['page']}):
\"\"\"
{chunk['text']}
\"\"\"
"""
    try:
        options = {
            "temperature": temperature,
            "top_p": 0.95,
            "top_k": 40,
        }
        response = ollama.chat(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            options=options
        )
        content = response["message"]["content"]

        if "INSTRUCTION:" not in content or "RESPONSE:" not in content:
            return None, None

        instruction = content.split("INSTRUCTION:")[1].split("RESPONSE:")[0].strip()
        answer = content.split("RESPONSE:")[1].strip().split("\n")[0].strip()

        instruction = re.sub(r"<think>.*?</think>", "", instruction, flags=re.DOTALL).strip()
        answer = re.sub(r"<think>.*?</think>", "", answer, flags=re.DOTALL).strip()

        # Length validation with tolerance
        q_words = len(instruction.split())
        a_words = len(answer.split())
        if q_words > MAX_WORDS_QA_ACCEPT or a_words > MAX_WORDS_QA_ACCEPT:
            return None, None

        # Reject generic questions
        generic_patterns = [
            r"^what is the (description|content|section|text|page|manual)",
            r"^what does this (section|page|text) (say|describe|contain|cover)",
            r"^what is (included|covered|discussed) in this",
            r"^what is this (section|chapter|page) about",
        ]
        for pat in generic_patterns:
            if re.match(pat, instruction.lower()):
                return None, None

        return instruction, answer
    except Exception as e:
        return None, None

# 5. Consistency verification
def verify_qa_consistency(chunk: dict, question: str, answer: str, model: str = "qwen3-coder") -> str:
    prompt = f"""You are a QA dataset quality checker.

Check if the QUESTION and ANSWER below are:
1. Directly supported by the CONTENT.
2. Specific and technical (not vague).
3. Accurate according to the CONTENT.

Reply with ONLY one word:
- "high" → accurate, specific, grounded.
- "low"  → vague or partially supported.
- "rejected" → wrong, hallucinated, or unanswerable.

CONTENT:
\"\"\"
{chunk['text']}
\"\"\"

QUESTION: {question}
ANSWER: {answer}

Verdict:"""
    try:
        response = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}])
        verdict = response["message"]["content"].strip().lower()
        verdict = re.sub(r"<think>.*?</think>", "", verdict, flags=re.DOTALL).strip()
        if verdict in ("high", "low", "rejected"):
            return verdict
        return "low"
    except Exception:
        return "low"

# 6. Similarity filter with optional logging
def is_similar_to_existing(new_q: str, existing_qs: List[str], threshold: float = 0.7) -> bool:
    new_words = set(new_q.lower().split())
    for old_q in existing_qs:
        old_words = set(old_q.lower().split())
        if not new_words or not old_words:
            continue
        intersection = new_words & old_words
        union = new_words | old_words
        jaccard = len(intersection) / len(union)
        if jaccard > threshold:
            return True
    return False

# 7. Save as JSONL
def save_to_jsonl(pairs: List[QAPair], output_file: str):
    stats = {"high": 0, "low": 0, "rejected": 0}
    with open(output_file, "w", encoding="utf-8") as f:
        for pair in pairs:
            stats[pair.confidence] += 1
            if pair.confidence != "rejected":
                f.write(json.dumps(asdict(pair), ensure_ascii=False) + "\n")
    print(f"\n📊 Final results:")
    print(f"   ✅ High confidence : {stats['high']}")
    print(f"   ⚠️  Low confidence  : {stats['low']}")
    print(f"   ❌ Rejected        : {stats['rejected']}")

# Main pipeline
def generate_dataset(
    pdf_path: str,
    model: str = "qwen3-coder",
    output_file: str = "dataset.jsonl",
    max_chunks: Optional[int] = None,
    max_retries: int = 5,                    # Increased
    max_qa_pairs: Optional[int] = None,
    temperature: float = 0.9,
    similarity_threshold: float = 0.7,
):
    print("📄 Extracting text from PDF...")
    pages = extract_text_from_pdf(pdf_path)

    print("✂️  Splitting into semantic chunks...")
    chunks = split_into_chunks(pages)
    initial_chunk_count = len(chunks)
    chunks = [c for c in chunks if is_valid_chunk(c["text"])]
    print(f"   Valid chunks: {len(chunks)} of {initial_chunk_count}")

    if max_chunks:
        chunks = chunks[:max_chunks]

    print(f"🧠 Generating Q&A for {len(chunks)} chunks using model: {model}")
    pairs: List[QAPair] = []
    existing_questions: List[str] = []

    # Diagnostic counters
    stats = {
        "generation_failed": 0,
        "too_many_words": 0,
        "similar_rejected": 0,
        "low_confidence": 0,
        "high_confidence": 0,
        "max_retries_exceeded": 0,
    }

    for chunk in tqdm(chunks, desc="Processing", unit="chunk"):
        if max_qa_pairs is not None and len(pairs) >= max_qa_pairs:
            print(f"\n🏁 Limit of {max_qa_pairs} QA pairs reached.")
            break

        question = answer = None
        confidence = "rejected"
        success = False

        for attempt in range(max_retries):
            question, answer = generate_qa(chunk, model, temperature)
            if not question or not answer:
                stats["generation_failed"] += 1
                continue

            q_words = len(question.split())
            a_words = len(answer.split())
            if q_words > MAX_WORDS_QA_ACCEPT or a_words > MAX_WORDS_QA_ACCEPT:
                stats["too_many_words"] += 1
                continue

            if is_similar_to_existing(question, existing_questions, similarity_threshold):
                stats["similar_rejected"] += 1
                continue

            confidence = verify_qa_consistency(chunk, question, answer, model)
            if confidence == "high":
                stats["high_confidence"] += 1
                success = True
                break
            elif confidence == "low":
                stats["low_confidence"] += 1
                # Accept "low" if it's the last attempt
                if attempt == max_retries - 1:
                    success = True
                    break
            else:  # rejected
                continue

        if not success:
            stats["max_retries_exceeded"] += 1
            continue

        if question and answer and confidence != "rejected":
            pairs.append(QAPair(
                question=question,
                answer=answer,
                source_chunk=chunk["text"][:300],
                page_hint=f"page_{chunk['page']}",
                confidence=confidence,
            ))
            existing_questions.append(question)

    # Display diagnostics
    print("\n📈 Generation diagnostics:")
    print(f"   Generation failures (format): {stats['generation_failed']}")
    print(f"   Exceeded word limit: {stats['too_many_words']}")
    print(f"   Rejected by similarity: {stats['similar_rejected']}")
    print(f"   High confidence obtained: {stats['high_confidence']}")
    print(f"   Low confidence accepted: {stats['low_confidence']}")
    print(f"   Chunks unsuccessful after {max_retries} attempts: {stats['max_retries_exceeded']}")

    save_to_jsonl(pairs, output_file)
    print(f"\n✅ Dataset saved to: {output_file} ({len(pairs)} examples processed)")

if __name__ == "__main__":
    generate_dataset(
        "CAT-4083-UK.pdf",
        model="qwen3-coder",          # or "qwen3-coder", "mistral", etc.
        output_file="dataset_CAT-4083-UK.jsonl",
        max_chunks=200,            # Increase if needed
        max_retries=5,
        max_qa_pairs=200,
        temperature=0.9,
        similarity_threshold=0.75,  # slightly more tolerant
    )

## Next Steps
After running the notebook, you will get a `dataset.jsonl` file containing high-quality QA pairs. You can then:
- Filter only high-confidence pairs.
- Convert to other formats (e.g., CSV, Hugging Face Dataset).
- Use for fine-tuning or evaluation.

**Note:** Make sure Ollama is running and the specified model (e.g., `qwen3-coder`) is pulled before execution.